In [ ]:
# -*- coding: utf-8 -*-
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#    http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
# implied.
# See the License for the specific language governing permissions and
# limitations under the License.
#

# Hyperparameter Tuning for Deep Learning Models

>Important Note!: Running cca 5min without GPU

The keras tuner is a library that helps you pick the optimal set of hyperparameters for your tensorflow program.
Hyperparameter tuning, or hyper-tuning, is the process of selecting the right set of hyperparameters for your machine learning (ml) application.

>Important Concepts:
>- search space: the range of hyperparameter values to explore.
?- objective: the metric that is optimized during the search (e.g., val_accuracy).
>- early stopping: a regularization method to stop training when performance stops improving.

>Reference Link: https://keras.io/guides/keras_tuner/getting_started/

### Setup

Uncomment the pip install command if keras tuner is not installed.

In [ ]:
# !pip install tensorflow keras-tuner -q

In [ ]:
import tensorflow as tf
from matplotlib import pyplot as plt
from tensorflow import keras
import pandas as pd

In [ ]:
import keras_tuner as kt

In [ ]:
# this function plots training and validation loss and accuracy over epochs.
# it is useful to visualize the learning process and determine if the model is overfitting
def plot_graphs(history):
    plt.subplots(figsize=(15, 5))

    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'])
    plt.plot(history.history['val_loss'])
    plt.title('model loss')
    plt.ylabel('loss')
    plt.xlabel('epoch')
    plt.legend(['train', 'val'], loc='upper left')

    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title('model accuracy')
    plt.ylabel('accuracy')
    plt.xlabel('epoch')
    plt.legend(['train', 'val'], loc='upper left')

    plt.show()

## Load MNIST

The fashion mnist dataset is loaded and preprocessed. The images are normalized to the range [0, 1] by dividing by 255.

In [ ]:
(img_train, label_train), (img_test, label_test) = keras.datasets.fashion_mnist.load_data()

# normalize pixel values between 0 and 1
img_train = img_train.astype('float32') / 255.0
img_test = img_test.astype('float32') / 255.0


## Baseline Architecture

A simple deep neural network (dnn) model is defined. This baseline model flattens the image, applies a dense layer with a specified number of units and activation, optionally applies dropout, and finally outputs predictions with a softmax layer.

Key term:
- `dropout`: a regularization technique that randomly sets a fraction of the inputs to 0 during training. For example, with dropout rate 0.25, each neuron is set to 0 with probability 0.25 during training.

In [ ]:
from keras import layers

def baseline_model(units, activation, dropout, lr):
    model = keras.Sequential()
    model.add(layers.Flatten())
    model.add(layers.Dense(units=units, activation=activation))
    if dropout:
        model.add(layers.Dropout(rate=0.25))
    model.add(layers.Dense(10, activation="softmax"))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"],
    )
    return model


## Define Hyperparameter Space

The function build_model defines the hyperparameter space to explore.
Key hyperparameters include:
- units: the number of neurons in the dense layer.
- activation: the activation function (e.g., relu, tanh).
- dropout: whether to use dropout.
- lr: learning rate, which is searched on a logarithmic scale.

In [ ]:
def build_model(hp):
    units = hp.Int("units", min_value=32, max_value=512, step=32)
    activation = hp.Choice("activation", ["relu", "tanh"])
    dropout = hp.Boolean("dropout")

    lr = hp.Float("lr", min_value=1e-4, max_value=1e-2, sampling="log")
    # call existing model-building code with the hyperparameter values.
    model = baseline_model(
        units=units, activation=activation, dropout=dropout, lr=lr
    )
    return model


build_model(kt.HyperParameters())

## Instantiate the tuner and perform hypertuning

The keras tuner currently supports several tuners (`RandomSearch`, `Hyperband`, `BayesianOptimization`, `Sklearn`). In this example, the `Hyperband` tuner is used.
- hyperband: an algorithm that efficiently explores hyperparameter space by adapting the number of epochs allocated.

The tuner searches for hyperparameters that maximize the validation accuracy.

In [ ]:
# define tuner and parameters
tuner = kt.Hyperband(
    build_model,
    objective='val_accuracy',
    max_epochs=10,
    factor=3,
    directory='my_dir',
    project_name='intro_to_kt'
)

# define callbacks, including early stopping to prevent overfitting
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

# print a summary of the search space
tuner.search_space_summary()

### Run Tuner

The tuner is run on the training data with a validation split.
It will search for the best hyperparameter configuration over a number of epochs.

In [ ]:
tuner.search(img_train, label_train, epochs=10, validation_split=0.2, callbacks=[stop_early])

## Explore Hyperparameters

After the search, the best hyperparameters can be extracted and examined.
The tuner also provides a summary of the best models and their hyperparameter values.

In [ ]:
# get the optimal hyperparameters (best trial)
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

# display the best hyperparameters from the top 5 trials as a dataframe
pd.DataFrame([_.values for _ in tuner.get_best_hyperparameters(5)])

In [ ]:
# retrieve the best models from the tuner
models = tuner.get_best_models(num_models=2)
best_model = models[0]
best_model.summary()

In [ ]:
# print a summary of the tuner results
tuner.results_summary()

## Train the Model

### Determine the optimal number of epochs to train the model with the hyperparameters obtained from search

In [ ]:
# build a model using the best hyperparameters
model = tuner.hypermodel.build(best_hps)
# train the model on the training data with a validation split
best_model_history = model.fit(img_train, label_train, epochs=10, validation_split=0.2)

# find the best epoch based on the highest validation accuracy
val_acc_per_epoch = best_model_history.history['val_accuracy']
best_epoch = val_acc_per_epoch.index(max(val_acc_per_epoch)) + 1
print(f'Best epoch: {best_epoch}')

In [ ]:
# plot training graphs for the best model history
plot_graphs(best_model_history)

### Retrain the model with optimal number of epochs

After determining the best number of epochs, the model is retrained on the full training data.

In [ ]:
hypermodel = tuner.hypermodel.build(best_hps)

# retrain the model using the best number of epochs
history = hypermodel.fit(img_train, label_train, epochs=best_epoch, validation_split=0.2)

In [ ]:
# plot training graphs
plot_graphs(best_model_history)

### Evaluate the Model

The retrained model is evaluated on the test set to determine its final performance.

In [ ]:
eval_result = hypermodel.evaluate(img_test, label_test)
print(f'[test loss, test accuracy]: {eval_result}')

The `my_dir/intro_to_kt` directory contains detailed logs and checkpoints for every trial (model configuration) run during the hyperparameter search.
If you re-run the hyperparameter search, the keras tuner uses the existing state from these logs to resume the search.
To disable this behavior, pass an additional overwrite=True argument while instantiating the tuner.
